This experiment is dedicated to test $\hat R_\nu$  on different multivariate normal distribution with the variance for each dimension varies, and AR(1) covariance.

In [1]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 145.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 MB 32.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


**Package Import and Other Setups**

In [1]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [3]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/BJX_Utility.py'
with open(utility_link) as f: exec(f.read())
plot_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/D100_Utility.py'
with open(plot_link) as f: exec(f.read())

Mounted at /content/drive


**Hyperparm Setups**

In [4]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [5]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**Experiment Part**

In [6]:
num_dim = 100
rho = 0.5

# Mean
mu = jnp.full(num_dim, 0.0)

# Base correlation matrix
corr = jnp.array([
    [rho ** abs(i - j) for j in range(num_dim)]
    for i in range(num_dim)
])

# --------------------------------------------------
# Randomly select dimensions for different variances
# --------------------------------------------------

base_key = random.PRNGKey(42)

# Split keys so the selections are reproducible
key_1000, key_100 = random.split(base_key)

# Randomly choose 10 distinct dimensions for variance 1000
selected_1000 = random.choice(
    key_1000,
    num_dim,
    shape=(10,),
    replace=False
)

# Choose another 10 distinct dimensions for variance 100
# while excluding the first 10
remaining = jnp.setdiff1d(
    jnp.arange(num_dim),
    selected_1000
)

selected_100 = random.choice(
    key_100,
    remaining,
    shape=(10,),
    replace=False
)

# --------------------------------------------------
# Construct variance vector
# --------------------------------------------------

variances = jnp.ones(num_dim)

variances = variances.at[selected_1000].set(1000.0)
variances = variances.at[selected_100].set(100.0)

# Standard deviations
std = jnp.sqrt(variances)
cov = std[:, None] * corr * std[None, :]


target = tfd.MultivariateNormalFullCovariance(
    loc=mu,
    covariance_matrix=cov
)

init_step_size = 0.5

def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [8]:
#simulation part:
AR_MSE_c_list = []
AR_MSE_n_list = []
AR_RHat_c_list = []
AR_RHat_n_list = []

In [9]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, AR_RHat_c_list,AR_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, AR_RHat_n_list,AR_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 2001.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.04335308074951172
Naive initialization. Warmup Length: 10; mean of MSE is: 0.008903519250452518
Simulation Start: 2316.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.009986018761992455
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0057631563395261765
Simulation Start: 2363.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.006824294570833445
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0038703836034983397
Simulation Start: 2397.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0054476624354720116
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0031420632731169462
Simulation Start: 2451.5 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.004292147699743509
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00252829329110682
Simulation Start: 2479.7 MB
Constrained initiali

**Save to Dataframe and File**

In [10]:
MSE_c_df = pd.DataFrame(AR_MSE_c_list)
R_Hat_c_df = pd.DataFrame(AR_RHat_c_list)

MSE_n_df = pd.DataFrame(AR_MSE_n_list)
R_Hat_n_df = pd.DataFrame(AR_RHat_n_list)

In [11]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/AR_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/AR_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/AR_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/AR_Rhat_n.pkl"
)

In [12]:
from google.colab import runtime
runtime.unassign()

**Controlled Experiment-With No Rotation, just the variance**

In [ ]:
num_dim = 100

# Mean array
mu = jnp.full(num_dim, 0.0)

base_key = random.PRNGKey(42)

# Split keys so the selections are reproducible
key_1000, key_100 = random.split(base_key)

# Randomly choose 10 distinct dimensions for variance 1000
selected_1000 = random.choice(
    key_1000,
    num_dim,
    shape=(10,),
    replace=False
)

# Choose another 10 distinct dimensions for variance 100
# while excluding the first 10
remaining = jnp.setdiff1d(
    jnp.arange(num_dim),
    selected_1000
)

selected_100 = random.choice(
    key_100,
    remaining,
    shape=(10,),
    replace=False
)

# Variance = 1 for all dimensions initially
variances = jnp.full(num_dim, 1.0)

# Assign randomly selected dimensions
variances = variances.at[selected_1000].set(1000.0)
variances = variances.at[selected_100].set(100.0)

# Diagonal covariance matrix
cov = jnp.diag(variances)

target = tfd.MultivariateNormalFullCovariance(
    loc=jnp.array(mu),
    covariance_matrix=cov
)

init_step_size = 0.5

def target_log_prob_fn(x):
    return target.log_prob(x)

def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
Controll_MSE_c_list = []
Controll_MSE_n_list = []
Controll_RHat_c_list = []
Controll_RHat_n_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Controll_RHat_c_list, Controll_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Controll_RHat_n_list, Controll_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1704.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.008116121403872967
Naive initialization. Warmup Length: 10; mean of MSE is: 0.003715771483257413
Simulation Start: 1953.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0057884822599589825
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0027356361970305443
Simulation Start: 1967.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.003920773044228554
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0018808534368872643
Simulation Start: 1982.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0031254147179424763
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0015257771592587233
Simulation Start: 1998.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.002421633107587695
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0012467099586501718
Simulation Start: 2022.3 MB
Constrained init

In [ ]:
MSE_c_df = pd.DataFrame(Controll_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Controll_RHat_c_list)

MSE_n_df = pd.DataFrame(Controll_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Controll_RHat_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Controll_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Controll_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Controll_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Controll_Rhat_n.pkl"
)